# ResNet-18 vs EfficientNet-B0 Comparison

This notebook directly compares the existing evaluation results and parameter counts. It does not train, evaluate, or create any datasets.

In [2]:
import json
from pathlib import Path

import pandas as pd
from torch import nn
from torchvision.models import efficientnet_b0, resnet18

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS_DIR = PROJECT_ROOT / 'results'
RESNET_METRICS_PATH = RESULTS_DIR / 'resnet18_test_metrics.json'
EFFICIENTNET_METRICS_PATH = RESULTS_DIR / 'efficientnet_b0_test_metrics.json'

assert RESNET_METRICS_PATH.exists()
assert EFFICIENTNET_METRICS_PATH.exists()

In [3]:
resnet_metrics = json.loads(RESNET_METRICS_PATH.read_text(encoding='utf-8'))
efficientnet_metrics = json.loads(EFFICIENTNET_METRICS_PATH.read_text(encoding='utf-8'))

metric_labels = {
    'test_accuracy': 'Test Accuracy',
    'precision': 'Precision',
    'recall': 'Recall',
    'f1_score': 'F1-score',
    'top5_accuracy': 'Top-5 Accuracy',
}

comparison_table = pd.DataFrame({
    'Metric': [metric_labels[key] for key in metric_labels],
    'ResNet-18': [resnet_metrics[key] for key in metric_labels],
    'EfficientNet-B0': [efficientnet_metrics[key] for key in metric_labels],
})
comparison_table.style.format({'ResNet-18': '{:.6f}', 'EfficientNet-B0': '{:.6f}'})

,Metric,ResNet-18,EfficientNet-B0
0,Test Accuracy,0.686000,0.742000
1,Precision,0.725493,0.779000
2,Recall,0.686000,0.742000
3,F1-score,0.679440,0.737577
4,Top-5 Accuracy,0.904000,0.928000


In [ ]:
winner_rows = []
for key, label in metric_labels.items():
    resnet_value = resnet_metrics[key]
    efficientnet_value = efficientnet_metrics[key]
    if resnet_value > efficientnet_value:
        winner = 'ResNet-18'
    elif efficientnet_value > resnet_value:
        winner = 'EfficientNet-B0'
    else:
        winner = 'Tie'
    winner_rows.append({'Metric': label, 'Higher value': winner})

metric_winners = pd.DataFrame(winner_rows)
metric_winners

,Metric,Higher value
0,Test Accuracy,EfficientNet-B0
1,Precision,EfficientNet-B0
2,Recall,EfficientNet-B0
3,F1-score,EfficientNet-B0
4,Top-5 Accuracy,EfficientNet-B0


In [4]:
comparison_table.to_csv(RESULTS_DIR / 'model_comparison.csv', index=False)
comparison_table

,Metric,ResNet-18,EfficientNet-B0
0,Test Accuracy,0.686000,0.742000
1,Precision,0.725493,0.779000
2,Recall,0.686000,0.742000
3,F1-score,0.679440,0.737577
4,Top-5 Accuracy,0.904000,0.928000


## Parameter Counts

The models below are constructed with `weights=None` and a 100-class classifier only to count parameters. No checkpoint inference or training is performed.

In [5]:
num_classes = 100
resnet_model = resnet18(weights=None)
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, num_classes)
efficientnet_model = efficientnet_b0(weights=None)
efficientnet_model.classifier[1] = nn.Linear(efficientnet_model.classifier[1].in_features, num_classes)

def parameter_counts(model):
    parameters = list(model.parameters())
    return sum(parameter.numel() for parameter in parameters), sum(
        parameter.numel() for parameter in parameters if parameter.requires_grad
    )

resnet_total, resnet_trainable = parameter_counts(resnet_model)
efficientnet_total, efficientnet_trainable = parameter_counts(efficientnet_model)
parameter_table = pd.DataFrame([
    {'Model': 'ResNet-18', 'Total Parameters': resnet_total, 'Trainable Parameters': resnet_trainable},
    {'Model': 'EfficientNet-B0', 'Total Parameters': efficientnet_total, 'Trainable Parameters': efficientnet_trainable},
])
parameter_table

comparison_export = comparison_table.copy()
comparison_export['ResNet-18 Total Parameters'] = resnet_total
comparison_export['EfficientNet-B0 Total Parameters'] = efficientnet_total
comparison_export.to_csv(RESULTS_DIR / 'model_comparison.csv', index=False)


In [ ]:
summary_table = pd.DataFrame({
    'Comparison': [
        'Higher Test Accuracy',
        'Higher Precision',
        'Higher Recall',
        'Higher F1-score',
        'Higher Top-5 Accuracy',
        'Fewer Total Parameters',
    ],
    'Result': [
        metric_winners.loc[metric_winners['Metric'] == 'Test Accuracy', 'Higher value'].iloc[0],
        metric_winners.loc[metric_winners['Metric'] == 'Precision', 'Higher value'].iloc[0],
        metric_winners.loc[metric_winners['Metric'] == 'Recall', 'Higher value'].iloc[0],
        metric_winners.loc[metric_winners['Metric'] == 'F1-score', 'Higher value'].iloc[0],
        metric_winners.loc[metric_winners['Metric'] == 'Top-5 Accuracy', 'Higher value'].iloc[0],
        'ResNet-18' if resnet_total < efficientnet_total else 'EfficientNet-B0' if efficientnet_total < resnet_total else 'Tie',
    ],
})
summary_table

,Comparison,Result
0,Higher Test Accuracy,EfficientNet-B0
1,Higher Precision,EfficientNet-B0
2,Higher Recall,EfficientNet-B0
3,Higher F1-score,EfficientNet-B0
4,Higher Top-5 Accuracy,EfficientNet-B0
5,Fewer Total Parameters,EfficientNet-B0


## Conclusion

Under the experimental conditions used in this project, EfficientNet-B0 achieved the stronger overall classification performance: it has the higher value for Test Accuracy, Precision, Recall, F1-score, and Top-5 Accuracy in the existing evaluation results. This conclusion is limited to these experiments and does not claim that the architecture is universally better.